# 데이터 변환

## 기본 파생변수 생성

In [1]:
import pandas as pd
import numpy as np

In [2]:
# 데이터 생성
sales_data = pd.DataFrame({
    'product': ['노트북', '마우스', '키보드', '모니터', '스피커'],
    'price': [1200000, 25000, 80000, 350000, 120000],
    'quantity': [10, 50, 30, 15, 25],
    'cost': [800000, 15000, 50000, 200000, 80000]
})

print(sales_data)

  product    price  quantity    cost
0     노트북  1200000        10  800000
1     마우스    25000        50   15000
2     키보드    80000        30   50000
3     모니터   350000        15  200000
4     스피커   120000        25   80000


### 1. 새로운 컬럼 생성

In [ ]:
# 총 매출액 계산
# 새로운 컬럼: df['새로운컬럼명'] = 계산식
sales_data['total_sales'] = sales_data['price'] * sales_data['quantity']
print(sales_data)

  product    price  quantity    cost  total_sales
0     노트북  1200000        10  800000     12000000
1     마우스    25000        50   15000      1250000
2     키보드    80000        30   50000      2400000
3     모니터   350000        15  200000      5250000
4     스피커   120000        25   80000      3000000


In [4]:
# 총 이익 계산
sales_data['profit'] = (sales_data['price'] - sales_data['cost']) * sales_data['quantity']
print(sales_data)

  product    price  quantity    cost  total_sales   profit
0     노트북  1200000        10  800000     12000000  4000000
1     마우스    25000        50   15000      1250000   500000
2     키보드    80000        30   50000      2400000   900000
3     모니터   350000        15  200000      5250000  2250000
4     스피커   120000        25   80000      3000000  1000000


In [7]:
# 이익률 계산
sales_data['profit_rate'] = (sales_data['profit'] / sales_data['total_sales']) * 100
print(sales_data[['product', 'total_sales', 'profit', 'profit_rate']])

  product  total_sales   profit  profit_rate
0     노트북     12000000  4000000    33.333333
1     마우스      1250000   500000    40.000000
2     키보드      2400000   900000    37.500000
3     모니터      5250000  2250000    42.857143
4     스피커      3000000  1000000    33.333333


### 2. assign() 메서드 사용
- DataFrame에 새로운 열을 추가하거나 기존 열을 수정한 새로운 DataFrame을 반환

In [ ]:
# assign() 메서드 사용
# df.assign(컬럼명=값)
sales_with_rank = sales_data.assign(
    sales_rank=sales_data['total_sales'].rank(ascending=False)
)
print(sales_with_rank[['product', 'total_sales', 'sales_rank']])

  product  total_sales  sales_rank
0     노트북     12000000         1.0
1     마우스      1250000         5.0
2     키보드      2400000         4.0
3     모니터      5250000         2.0
4     스피커      3000000         3.0


### apply() 메서드를 활용한 사용자 정의 함수 적용

In [10]:
# 제품 등급 분류 함수 정의
def classify_product_grade(row):
    if row['profit_rate'] >= 40:
        return 'A급'
    elif row['profit_rate'] >= 35:
            return 'B급'
    else:
        return 'C급'


### 1. DataFrame에 함수 적용

In [13]:
# apply를 사용하여 제품 등급을 생성
# df.apply(함수, axis)
# axis: 0(행, row), 1(열, column)
sales_data['grade'] = sales_data.apply(classify_product_grade, axis=1)
print(sales_data)

  product    price  quantity    cost  total_sales   profit  profit_rate grade
0     노트북  1200000        10  800000     12000000  4000000    33.333333    C급
1     마우스    25000        50   15000      1250000   500000    40.000000    A급
2     키보드    80000        30   50000      2400000   900000    37.500000    B급
3     모니터   350000        15  200000      5250000  2250000    42.857143    A급
4     스피커   120000        25   80000      3000000  1000000    33.333333    C급


### 2. 람다 함수 사용

In [15]:
# 람다 함수를 사용한 가격 구간 분류

'''
   def price_category_fn(row):
    if row['price'] >= 300000:
        return '고가'
    elif row['price'] >= 100000:
        return '중가'
    else:
        return '저가'
'''

sales_data['price_category'] = sales_data['price'].apply(
    lambda x: '고가' if x >=300000 else '중가' if x >= 100000 else '저가'
)
print(sales_data[['product', 'profit_rate', 'grade', 'price_category']])

  product  profit_rate grade price_category
0     노트북    33.333333    C급             고가
1     마우스    40.000000    A급             저가
2     키보드    37.500000    B급             저가
3     모니터    42.857143    A급             고가
4     스피커    33.333333    C급             중가


### 3. 컬럼별 통계 계산(axis=0)

In [16]:
column_stats = sales_data[['price', 'quantity', 'profit']].apply(np.mean)
print(f"평균값:\n{column_stats}")

평균값:
price        355000.0
quantity         26.0
profit      1730000.0
dtype: float64


## map을 활용한 데이터 변환

In [17]:
print(sales_data)

  product    price  quantity    cost  total_sales   profit  profit_rate grade  \
0     노트북  1200000        10  800000     12000000  4000000    33.333333    C급   
1     마우스    25000        50   15000      1250000   500000    40.000000    A급   
2     키보드    80000        30   50000      2400000   900000    37.500000    B급   
3     모니터   350000        15  200000      5250000  2250000    42.857143    A급   
4     스피커   120000        25   80000      3000000  1000000    33.333333    C급   

  price_category  
0             고가  
1             저가  
2             저가  
3             고가  
4             중가  


### 1. map - 딕셔너리

In [ ]:
# 카데코리 매핑 딕셔너리 생성
category_mapping = {
    '노트북': '컴퓨터',
    '마우스': '주변기기',
    '키보드': '주변기기',
    '모니터': '디스플레이',
    '스피커': '오디오'
}

grade_points = {
    'A급': 100, 
    'B급': 80, 
    'C급': 60
}

In [19]:
# map을 사용한 카케고리 변환
sales_data['product_category'] = sales_data['product'].map(category_mapping)
print(sales_data)

  product    price  quantity    cost  total_sales   profit  profit_rate grade  \
0     노트북  1200000        10  800000     12000000  4000000    33.333333    C급   
1     마우스    25000        50   15000      1250000   500000    40.000000    A급   
2     키보드    80000        30   50000      2400000   900000    37.500000    B급   
3     모니터   350000        15  200000      5250000  2250000    42.857143    A급   
4     스피커   120000        25   80000      3000000  1000000    33.333333    C급   

  price_category product_category  
0             고가              컴퓨터  
1             저가             주변기기  
2             저가             주변기기  
3             고가            디스플레이  
4             중가              오디오  


In [21]:
# 등급별 포인트 매핑
sales_data['grade_points'] = sales_data['grade'].map(grade_points)
print(sales_data[['product', 'product_category', 'grade', 'grade_points']])

  product product_category grade  grade_points
0     노트북              컴퓨터    C급            60
1     마우스             주변기기    A급           100
2     키보드             주변기기    B급            80
3     모니터            디스플레이    A급           100
4     스피커              오디오    C급            60


In [23]:
# 숫자 데이터에 함수 적용(소수점 반올림)
sales_data['profit_rate_rounded'] = sales_data['profit_rate'].map(lambda x: round(x,1))
print(sales_data[['product', 'profit_rate', 'profit_rate_rounded']])

  product  profit_rate  profit_rate_rounded
0     노트북    33.333333                 33.3
1     마우스    40.000000                 40.0
2     키보드    37.500000                 37.5
3     모니터    42.857143                 42.9
4     스피커    33.333333                 33.3


## 조건부 파생병수 생성((np.where, np.select)
- np.where: 간단한 이진 조건
- np.select: 다중 조건 분류

### 1. np.where

In [25]:
# np.where를 사용한 이진 분류
# 문법: np.where(조건condition, 참value_if_true, 거짓value_if_false)
sales_data['is_profitable'] = np.where(sales_data['profit'] > 1000000, '고수익', '일반수익')
print(sales_data[['product', 'profit', 'is_profitable']])

  product   profit is_profitable
0     노트북  4000000           고수익
1     마우스   500000          일반수익
2     키보드   900000          일반수익
3     모니터  2250000           고수익
4     스피커  1000000          일반수익


### np.select를 사용한 다중 조건 분류

In [27]:
conditions = [
    sales_data['total_sales'] >= 10000000,
    sales_data['total_sales'] >= 5000000,
    sales_data['total_sales'] >= 2000000
]

choices = [
    '대형거래', 
    '중형거래', 
    '소형거래'
]

sales_data['transaction_size'] = np.select(conditions, choices, default='미니거래')
print(sales_data[['product', 'total_sales', 'is_profitable']])

  product  total_sales is_profitable
0     노트북     12000000           고수익
1     마우스      1250000          일반수익
2     키보드      2400000          일반수익
3     모니터      5250000           고수익
4     스피커      3000000          일반수익


### 3. pd.cut을 사용한 구간 분할

In [28]:
# pd.cut(df['컬럼명'], 구간, 이름)
price_bins = [0, 50000, 200000, float('inf')]
price_labels = ['저가','중가', '고가']

sales_data['price_segment'] = pd.cut(sales_data['price'], bins=price_bins, labels=price_labels)
print(sales_data[['product', 'price', 'price_segment']])

  product    price price_segment
0     노트북  1200000            고가
1     마우스    25000            저가
2     키보드    80000            중가
3     모니터   350000            고가
4     스피커   120000            중가


## 집계 함수를 활용한 파생변수 생성

In [ ]:
# 제품 카테고리별 평균 수익률 계산
# 그룹별 변환
# 문법: df.groupby('컬럼명').transform('함수명')
sales_data['category_avg_profit_rate'] = sales_data.groupby('product_category')['profit_rate'].transform('mean')
print(sales_data[['product', 'profit_rate', 'category_avg_profit_rate']])

  product  profit_rate  category_avg_profit_rate
0     노트북    33.333333                 33.333333
1     마우스    40.000000                 38.750000
2     키보드    37.500000                 38.750000
3     모니터    42.857143                 42.857143
4     스피커    33.333333                 33.333333


In [34]:
# 제품 카테고리별 총 매출 합계 계산
sales_data['category_total_sales'] = sales_data.groupby('product_category')['total_sales'].transform('sum')
print(sales_data[['product', 'product_category', 'total_sales', 'category_total_sales']])

  product product_category  total_sales  category_total_sales
0     노트북              컴퓨터     12000000              12000000
1     마우스             주변기기      1250000               3650000
2     키보드             주변기기      2400000               3650000
3     모니터            디스플레이      5250000               5250000
4     스피커              오디오      3000000               3000000


In [38]:
# 제품 카테고리별 총 매출 대비 비중 계산
sales_data['sales_share_in_category'] = (sales_data['total_sales'] / sales_data
['category_total_sales']) * 100
print(sales_data[['product', 'total_sales', 'category_total_sales', 
'sales_share_in_category']])

  product  total_sales  category_total_sales  sales_share_in_category
0     노트북     12000000              12000000               100.000000
3     모니터      5250000               5250000               100.000000
4     스피커      3000000               3000000               100.000000
2     키보드      2400000               3650000                65.753425
1     마우스      1250000               3650000                34.246575


In [41]:
# 정렬 후 누적 합계 계산(매출 순위별 누적 매출)
# 'total_salse' 컬럼을 기준으로 내림차순 정렬
sales_sorted = sales_data.sort_values('total_sales', ascending=False)
print(sales_sorted[['product', 'total_sales']])

  product  total_sales
0     노트북     12000000
3     모니터      5250000
4     스피커      3000000
2     키보드      2400000
1     마우스      1250000


In [ ]:
# 누적합계: df['컬럼'].cumsum()
sales_sorted['cumulative_sales'] = sales_sorted['total_sales'].cumsum()
print(sales_sorted[['product','cumulative_sales', 'total_sales']])

  product  cumulative_sales  total_sales
0     노트북          12000000     12000000
3     모니터          17250000      5250000
4     스피커          20250000      3000000
2     키보드          22650000      2400000
1     마우스          23900000      1250000


In [40]:
#  누적 합계 비율
sales_sorted['cumulative_sales_ratio'] = (sales_sorted['cumulative_sales'] / sales_sorted
['total_sales'].sum()) * 100
print(sales_sorted[['product', 'total_sales', 'cumulative_sales', 'cumulative_sales_ratio']].round(1))

  product  total_sales  cumulative_sales  cumulative_sales_ratio
0     노트북     12000000          12000000                    50.2
3     모니터      5250000          17250000                    72.2
4     스피커      3000000          20250000                    84.7
2     키보드      2400000          22650000                    94.8
1     마우스      1250000          23900000                   100.0
